# LoopMind — Full Training (CNN + Transformer)
**Run location**: browser Colab (colab.research.google.com), T4 GPU  
**Data**: Google Drive → `MyDrive/LoopMind/slakh2100/`  
**Outputs**: Drive → `MyDrive/LoopMind/cache/` and `MyDrive/LoopMind/checkpoints/`

Run cells top to bottom. If session disconnects, re-run Cell 1–3, then Cell 5 with `--resume`.

In [ ]:
# ── Cell 1: Mount Drive + clone / update repo ─────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
REPO = '/content/LoopMind'

if not os.path.exists(REPO):
    !git clone https://github.com/ranxindeng-blip/LoopMind {REPO}
else:
    !cd {REPO} && git pull

# Paths on Drive
DATA_ROOT   = '/content/drive/MyDrive/LoopMind/slakh2100'
CACHE_DIR   = '/content/drive/MyDrive/LoopMind/cache'
CKPT_DIR    = '/content/drive/MyDrive/LoopMind/checkpoints'
LIBRARY_PATH = '/content/drive/MyDrive/LoopMind/cache/library.pt'

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(CKPT_DIR,  exist_ok=True)
print('Drive mounted. Paths ready.')

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
!pip install -q pretty_midi librosa soundfile gradio pydub pyyaml
print('Dependencies installed.')

In [ ]:
# ── Cell 3: Verify Slakh2100 data structure ───────────────────────────────────
import os

for split in ['train', 'validation']:
    split_dir = os.path.join(DATA_ROOT, split)
    if os.path.isdir(split_dir):
        tracks = [d for d in os.listdir(split_dir) if os.path.isdir(os.path.join(split_dir, d))]
        print(f'{split}: {len(tracks)} tracks')
    else:
        print(f'WARNING: {split_dir} not found!')

# Peek at one track
sample_track = os.path.join(DATA_ROOT, 'train', sorted(os.listdir(os.path.join(DATA_ROOT, 'train')))[0])
print(f'\nSample track: {sample_track}')
print('  MIDI stems:', os.listdir(os.path.join(sample_track, 'MIDI'))[:5])
print('  Audio stems:', os.listdir(os.path.join(sample_track, 'stems'))[:5])

In [ ]:
# ── Cell 4: Extract pairs + features (run once, cached to Drive) ──────────────
# Estimated time: ~20-40 min for 400 tracks
# Safe to re-run: already-cached .npy files are skipped automatically.

import sys
sys.path.insert(0, REPO)

from data.extract_pairs import extract_pairs
from data.features import extract_and_cache

MAX_TRACKS = 400

print(f'Scanning {MAX_TRACKS} tracks ...')
records = extract_pairs(DATA_ROOT, CACHE_DIR, max_tracks=MAX_TRACKS)
print(f'\nTotal records: {len(records)}')

print('\nExtracting features (piano roll + mel) ...')
features = extract_and_cache(records, CACHE_DIR)
print('\nFeature extraction complete.')

In [ ]:
# ── Cell 5: Train ─────────────────────────────────────────────────────────────
# First run  : leave --resume out
# After disconnect: add --resume flag to continue from last.pt

!cd {REPO} && python train.py \
    --data_root   {DATA_ROOT} \
    --cache_dir   {CACHE_DIR} \
    --ckpt_dir    {CKPT_DIR} \
    --max_tracks  400 \
    --epochs      100 \
    --batch_size  64 \
    --lr          1e-4 \
    --hidden      256 \
    --embed_dim   128 \
    --n_heads     4 \
    --n_layers    2

# To resume after a disconnect, replace the cell above with:
# !cd {REPO} && python train.py \
#     --data_root   {DATA_ROOT} \
#     --cache_dir   {CACHE_DIR} \
#     --ckpt_dir    {CKPT_DIR} \
#     --max_tracks  400 \
#     --epochs      100 \
#     --batch_size  64 \
#     --lr          1e-4 \
#     --hidden      256 \
#     --embed_dim   128 \
#     --n_heads     4 \
#     --n_layers    2 \
#     --resume

In [ ]:
# ── Cell 6: Evaluate ──────────────────────────────────────────────────────────
!cd {REPO} && python evaluate.py \
    --ckpt_path  {CKPT_DIR}/best.pt \
    --data_root  {DATA_ROOT} \
    --cache_dir  {CACHE_DIR} \
    --max_tracks 400

In [ ]:
# ── Cell 7: Build stem library ────────────────────────────────────────────────
!cd {REPO} && python build_library.py \
    --data_root    {DATA_ROOT} \
    --cache_dir    {CACHE_DIR} \
    --ckpt_path    {CKPT_DIR}/best.pt \
    --max_tracks   400 \
    --library_path {LIBRARY_PATH}

In [ ]:
# ── Cell 8: Launch Gradio demo ────────────────────────────────────────────────
# Uses the LEGACY MLP checkpoint (BabySlakh demo) — demo/app.py imports dual_encoder_mlp
# To wire up the new full model, update demo/app.py to load the new checkpoint.

import sys, torch
sys.path.insert(0, REPO)

from models.dual_encoder_mlp import DualEncoder

DEMO_CKPT    = f'{CKPT_DIR}/best.pt'   # swap to demo checkpoint path if needed
DEMO_LIBRARY = LIBRARY_PATH

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt   = torch.load(DEMO_CKPT, map_location=device, weights_only=False)
args   = ckpt['args']
model  = DualEncoder(hidden=args['hidden'], embed_dim=args['embed_dim'])
model.load_state_dict(ckpt['model'])

library = torch.load(DEMO_LIBRARY, map_location='cpu', weights_only=False)

from demo.app import launch
launch(model, library, device=device, share=True)